<a href="https://colab.research.google.com/github/Levan-Danelia/FRTB/blob/main/FRTB_FXVG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# =============================================================================
# Cell 1: Setup and Initial Data
# -----------------------------------------------------------------------------
# This cell loads the initial portfolio data and regulatory parameters. The output displays
# the starting positions, which corresponds to Steps 1 and 2 in the report.
# =============================================================================

# Import necessary libraries
import pandas as pd
import numpy as np

# --- Initial Portfolio Data ---
# This data represents the starting positions and their gross vega sensitivities.
portfolio_data = [
    {'position_id': 1, 'bucket': 'GBP^USD', 'tenor_str': '6M', 'gross_sensitivity': -162184},
    {'position_id': 2, 'bucket': 'GBP^USD', 'tenor_str': '6M', 'gross_sensitivity': 228192},
    {'position_id': 3, 'bucket': 'GBP^USD', 'tenor_str': '1Y', 'gross_sensitivity': 83909},
    {'position_id': 4, 'bucket': 'JPY^USD', 'tenor_str': '1Y', 'gross_sensitivity': -124872}
]

# --- Regulatory Parameters ---
TENOR_TO_YEARS = {'6M': 0.5, '1Y': 1.0}
VEGA_RISK_WEIGHT = 1.00  # 100% for FX Vega
RHO_DELTA_COMPONENT = 1.0 # For Vega risk factors in the same FX bucket
ALPHA = 0.01
GAMMA_CROSS_BUCKET = 0.60 # 60% for FX

# Create the initial DataFrame
df = pd.DataFrame(portfolio_data)

print("--- Steps 1 & 2: Initial Portfolio, Risk Factors, and Gross Sensitivities ---")
print("Each unique combination of 'bucket' and 'tenor_str' is a distinct risk factor.")
print("\n" + df[['position_id', 'bucket', 'tenor_str', 'gross_sensitivity']].to_string(index=False))

--- Steps 1 & 2: Initial Portfolio, Risk Factors, and Gross Sensitivities ---
Each unique combination of 'bucket' and 'tenor_str' is a distinct risk factor.

 position_id  bucket tenor_str  gross_sensitivity
           1 GBP^USD        6M            -162184
           2 GBP^USD        6M             228192
           3 GBP^USD        1Y              83909
           4 JPY^USD        1Y            -124872


In [3]:
# =============================================================================
# Cell 2: Step 3 - Net Sensitivities
# -----------------------------------------------------------------------------
# As per Article 325f(5), sensitivities for identical risk factors are netted.
# We group by the risk factor components (bucket, tenor) and sum the sensitivities.
# =============================================================================

df_net = df.groupby(['bucket', 'tenor_str']).agg(
    net_sensitivity=('gross_sensitivity', 'sum')
).reset_index()
df_net['tenor_years'] = df_net['tenor_str'].map(TENOR_TO_YEARS)

print("\n\n--- Step 3: Net Sensitivities ---")
print("Gross sensitivities for each unique risk factor are summed to get the net sensitivity.")
print("\n" + df_net[['bucket', 'tenor_str', 'net_sensitivity']].to_string(index=False))



--- Step 3: Net Sensitivities ---
Gross sensitivities for each unique risk factor are summed to get the net sensitivity.

 bucket tenor_str  net_sensitivity
GBP^USD        1Y            83909
GBP^USD        6M            66008
JPY^USD        1Y          -124872


In [4]:
# =============================================================================
# Cell 3: Step 4 - Weighted Sensitivities
# -----------------------------------------------------------------------------
# Each net sensitivity is multiplied by the regulatory risk weight (100% for FX Vega).
# We also calculate S_b, the sum of weighted sensitivities per bucket, for the final aggregation.
# =============================================================================

df_weighted = df_net.copy()
df_weighted['weighted_sensitivity'] = df_weighted['net_sensitivity'] * VEGA_RISK_WEIGHT

# Calculate Sb for each bucket
s_b_values = df_weighted.groupby('bucket')['weighted_sensitivity'].sum().reset_index()
s_b_values = s_b_values.rename(columns={'weighted_sensitivity': 'S_b'})
df_weighted_with_sb = pd.merge(df_weighted, s_b_values, on='bucket')

print("\n\n--- Step 4: Weighted Sensitivities ---")
print("Net sensitivities are multiplied by the 100% regulatory risk weight.")
print("\n" + df_weighted_with_sb[['bucket', 'tenor_str', 'net_sensitivity', 'weighted_sensitivity', 'S_b']].round(2).to_string(index=False))



--- Step 4: Weighted Sensitivities ---
Net sensitivities are multiplied by the 100% regulatory risk weight.

 bucket tenor_str  net_sensitivity  weighted_sensitivity       S_b
GBP^USD        1Y            83909               83909.0  149917.0
GBP^USD        6M            66008               66008.0  149917.0
JPY^USD        1Y          -124872             -124872.0 -124872.0


In [5]:
# =============================================================================
# Cell 4: Steps 5 & 6 - Correlation Coefficients (Medium Scenario)
# -----------------------------------------------------------------------------
# Per Article 325ay and 325aw, we determine the correlation parameters for aggregation.
# =============================================================================

# --- Intra-Bucket Correlation (rho_kl) for GBP^USD bucket ---
gbp_positions = df_weighted[df_weighted['bucket'] == 'GBP^USD'].to_dict('records')
t1, t2 = gbp_positions[0]['tenor_years'], gbp_positions[1]['tenor_years']
rho_maturity_comp = np.exp(-ALPHA * abs(t1 - t2) / min(t1, t2))
rho_kl_medium = RHO_DELTA_COMPONENT * rho_maturity_comp

# --- Cross-Bucket Correlation (gamma_bc) ---
gamma_bc_medium = GAMMA_CROSS_BUCKET

print("\n\n--- Steps 5 & 6: Correlation Coefficients (Medium Scenario) ---")
print("\n--- Intra-Bucket Correlation (Step 5) ---")
print("Calculated for the GBP^USD bucket with tenors 0.5Y and 1Y.")
print(f"Delta Component: {RHO_DELTA_COMPONENT:.2%}")
print(f"Option Maturity Component: {rho_maturity_comp:.3%}")
print(f"Final Intra-Bucket Correlation (ρ_kl): {rho_kl_medium:.3%}")

print("\n--- Cross-Bucket Correlation (Step 6) ---")
print("The uniform correlation for aggregating different FX buckets.")
print(f"Cross-Bucket Correlation (γ_bc): {gamma_bc_medium:.2%}")



--- Steps 5 & 6: Correlation Coefficients (Medium Scenario) ---

--- Intra-Bucket Correlation (Step 5) ---
Calculated for the GBP^USD bucket with tenors 0.5Y and 1Y.
Delta Component: 100.00%
Option Maturity Component: 99.005%
Final Intra-Bucket Correlation (ρ_kl): 99.005%

--- Cross-Bucket Correlation (Step 6) ---
The uniform correlation for aggregating different FX buckets.
Cross-Bucket Correlation (γ_bc): 60.00%


In [6]:
# =============================================================================
# Cell 5: Steps 7 & 8 - Aggregation (Medium Scenario)
# -----------------------------------------------------------------------------
# Weighted sensitivities are aggregated first within each bucket (K_b),
# and then across all buckets to get the final capital charge.
# =============================================================================

# --- Intra-Bucket Aggregation (Step 7) ---
k_b_values = {}
for bucket_name, group in df_weighted.groupby('bucket'):
    if len(group) == 1:
        # For single-factor buckets, K_b is the absolute weighted sensitivity
        k_b = abs(group['weighted_sensitivity'].iloc[0])
    else:
        # For multi-factor buckets (GBP^USD), apply the full formula
        ws1, ws2 = group['weighted_sensitivity'].iloc[0], group['weighted_sensitivity'].iloc[1]
        sum_ws_sq = ws1**2 + ws2**2
        cross_term = 2 * rho_kl_medium * ws1 * ws2
        k_b = np.sqrt(max(0, sum_ws_sq + cross_term))
    k_b_values[bucket_name] = k_b

print("\n\n--- Step 7: Intra-Bucket Aggregation (K_b) ---")
df_kb = pd.DataFrame(list(k_b_values.items()), columns=['Bucket', 'K_b_Capital'])
print(df_kb.round(2).to_string(index=False))


# --- Cross-Bucket Aggregation (Step 8) ---
k_b_list = list(k_b_values.values())
s_b_list = s_b_values['S_b'].tolist()

sum_kb_sq = sum(k**2 for k in k_b_list)
cross_term_sb = 2 * gamma_bc_medium * s_b_list[0] * s_b_list[1]

capital_medium = np.sqrt(max(0, sum_kb_sq + cross_term_sb))

print("\n--- Step 8: Cross-Bucket Aggregation ---")
print(f"Sum of Squares (Σ K_b^2): {sum_kb_sq:,.2f}")
print(f"Sum of Cross-Products (Σ γ_bc * S_b * S_c): {cross_term_sb:,.2f}")
print("----------------------------------------------------------")
print(f"Medium Scenario Capital: {capital_medium:,.2f}")



--- Step 7: Intra-Bucket Aggregation (K_b) ---
 Bucket  K_b_Capital
GBP^USD    149548.94
JPY^USD    124872.00

--- Step 8: Cross-Bucket Aggregation ---
Sum of Squares (Σ K_b^2): 37,957,901,992.47
Sum of Cross-Products (Σ γ_bc * S_b * S_c): -22,464,522,748.80
----------------------------------------------------------
Medium Scenario Capital: 124,472.40


In [7]:
# =============================================================================
# Cell 6: Step 9 - High and Low Correlation Scenarios
# -----------------------------------------------------------------------------
# The total capital is recalculated under stressed correlation assumptions.
# =============================================================================

def calculate_total_capital(rho_kl, gamma_bc, df_w, df_s, k_b_vals):
    # Recalculate K_b for GBP^USD under the new rho
    gbp_group = df_w[df_w['bucket'] == 'GBP^USD']
    ws1, ws2 = gbp_group['weighted_sensitivity'].iloc[0], gbp_group['weighted_sensitivity'].iloc[1]
    sum_ws_sq_gbp = ws1**2 + ws2**2
    cross_term_gbp = 2 * rho_kl * ws1 * ws2
    k_b_gbp_new = np.sqrt(max(0, sum_ws_sq_gbp + cross_term_gbp))

    # Get K_b for JPY^USD (it doesn't change)
    k_b_jpy = k_b_vals['JPY^USD']

    # Recalculate final capital
    sum_kb_sq_new = k_b_gbp_new**2 + k_b_jpy**2
    s_b_list = df_s['S_b'].tolist()
    cross_term_sb_new = 2 * gamma_bc * s_b_list[0] * s_b_list[1]

    return np.sqrt(max(0, sum_kb_sq_new + cross_term_sb_new))

# High Scenario Correlations
rho_kl_high = min(rho_kl_medium * 1.25, 1.0)
gamma_bc_high = min(gamma_bc_medium * 1.25, 1.0)

# Low Scenario Correlations
rho_kl_low = max(2 * rho_kl_medium - 1, 0.75 * rho_kl_medium)
gamma_bc_low = max(2 * gamma_bc_medium - 1, 0.75 * gamma_bc_medium)

# Calculate Capital for each scenario
capital_high = calculate_total_capital(rho_kl_high, gamma_bc_high, df_weighted, s_b_values, k_b_values)
capital_low = calculate_total_capital(rho_kl_low, gamma_bc_low, df_weighted, s_b_values, k_b_values)

# --- Display Results ---
scenario_data = {
    'Scenario': ['Medium', 'High', 'Low'],
    'Intra-Bucket (ρ)': [f"{rho_kl_medium:.3%}", f"{rho_kl_high:.3%}", f"{rho_kl_low:.3%}"],
    'Cross-Bucket (γ)': [f"{gamma_bc_medium:.2%}", f"{gamma_bc_high:.2%}", f"{gamma_bc_low:.2%}"],
    'Calculated Capital': [capital_medium, capital_high, capital_low]
}
df_scenarios = pd.DataFrame(scenario_data)

print("\n\n--- Step 9: Correlation Scenarios ---")
print("The capital is recalculated under stressed correlation assumptions.")
print("\nResulting Capital per Scenario:")
print(df_scenarios.round(2).to_string(index=False))



--- Step 9: Correlation Scenarios ---
The capital is recalculated under stressed correlation assumptions.

Resulting Capital per Scenario:
Scenario Intra-Bucket (ρ) Cross-Bucket (γ)  Calculated Capital
  Medium          99.005%           60.00%           124472.40
    High         100.000%           75.00%            99937.33
     Low          98.010%           45.00%           144911.31


In [8]:
# =============================================================================
# Cell 7: Step 10 - Final Charge Calculation
# -----------------------------------------------------------------------------
# The final capital requirement is the maximum of the three scenarios.
# =============================================================================

final_charge = df_scenarios['Calculated Capital'].max()
winning_scenario = df_scenarios.loc[df_scenarios['Calculated Capital'].idxmax()]['Scenario']

print("\n\n--- Step 10: Final Charge Calculation ---")
print(f"The final requirement is the maximum of the three scenarios.")
print("\n-------------------------------------------------")
print(f" Final FX Vega Capital Requirement: {final_charge:,.2f}")
print(f" (Driven by the {winning_scenario} Correlation Scenario)")
print("-------------------------------------------------")



--- Step 10: Final Charge Calculation ---
The final requirement is the maximum of the three scenarios.

-------------------------------------------------
 Final FX Vega Capital Requirement: 144,911.31
 (Driven by the Low Correlation Scenario)
-------------------------------------------------
